In [ ]:
!pip install pandas 
import pandas as pd

In [1]:
import sys
!{sys.executable} -m pip install yfinance
!pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [2]:
!pip install --user yfinance
import yfinance as yf

Looking in links: /usr/share/pip-wheels


In [3]:
tesla = yf.Ticker("TSLA")

In [4]:
tesla_data = tesla.history(period="max")

In [5]:
tesla_data.reset_index(inplace=True)

In [6]:
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [29]:
!pip install urllib3
!pip install requests 
import os
os.environ['HTTP_PROXY'] = ''
os.environ['HTTPS_PROXY'] = ''
os.environ['http_proxy'] = ''
os.environ['https_proxy'] = ''
import urllib3
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Disable SSL warning messages
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Function to download HTML content with error handling and retries
def download_html_with_retry(url, max_retries=3, delay=2):
    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1} to download data...")
            # Add timeout and headers to improve connection reliability
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            response = requests.get(url, verify=False, timeout=30, headers=headers)
            response.raise_for_status()  # Raise an exception for bad status codes
            return response.text
        except requests.exceptions.ConnectionError as e:
            print(f"Connection error on attempt {attempt + 1}: {e}")
        except requests.exceptions.Timeout as e:
            print(f"Timeout error on attempt {attempt + 1}: {e}")
        except requests.exceptions.RequestException as e:
            print(f"Request error on attempt {attempt + 1}: {e}")
        
        if attempt < max_retries - 1:
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)
    
    return None

# Download HTML content with error handling
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = download_html_with_retry(url)

if html_data is None:
    print("Failed to download data after multiple attempts.")
    print("Please check your internet connection or try again later.")
    # Create empty DataFrame as fallback
    tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
else:
    try:
        # Parse HTML tree
        soup = BeautifulSoup(html_data, "html.parser")
        
        # Extract table rows with error handling
        tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
        tables = soup.find_all("table")
        
        if len(tables) < 2:
            print("Error: Expected table not found in the HTML content.")
            tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
        else:
            tbody = tables[1].find("tbody")
            if tbody is None:
                print("Error: Table body not found.")
                tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
            else:
                for row in tbody.find_all("tr"):
                    col = row.find_all("td")
                    if len(col) >= 2:  # Ensure we have at least 2 columns
                        date = col[0].text.strip()
                        revenue = col[1].text.strip()
                        tesla_revenue = pd.concat([tesla_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})], ignore_index=True)
                
                # Clean commas and dollar signs
                tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(r',|\$', "", regex=True)
                tesla_revenue.dropna(inplace=True)
                tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]
                
                print("Data successfully extracted and cleaned.")
    
    except Exception as e:
        print(f"Error processing HTML data: {e}")
        tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

# Display results
if not tesla_revenue.empty:
    print("Last 5 rows of Tesla revenue data:")
    print(tesla_revenue.tail())
else:
    print("No data available to display.")





Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Attempt 1 to download data...
Connection error on attempt 1: HTTPSConnectionPool(host='cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud', port=443): Max retries exceeded with url: /IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x703e6717d940>: Failed to establish a new connection: [Errno 111] Connection refused'))
Retrying in 2 seconds...
Attempt 2 to download data...
Connection error on attempt 2: HTTPSConnectionPool(host='cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud', port=443): Max retries exceeded with url: /IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm (Caused by NewConnectionError('<urllib3.con

In [33]:
# Create Ticker object for GameStop
gme = yf.Ticker("GME")

# Extract historical market data
gme_data = gme.history(period="max")

# Reset index
gme_data.reset_index(inplace=True)

# Display first 5 rows
gme_data.head()

Failed to get ticker 'GME' reason: Failed to perform, curl: (7) Failed to connect to query2.finance.yahoo.com port 443 after 0 ms: Could not connect to server. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$GME: possibly delisted; no timezone found


,Date,Open,High,Low,Close,Adj Close,Volume


In [1]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Historical Share Price", "Historical Revenue"), vertical_spacing = .3)
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date, infer_datetime_format=True), y=stock_data_specific.Close.astype("float"), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date, infer_datetime_format=True), y=revenue_data_specific.Revenue.astype("float"), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
print("--- Question 1: Tesla Data First 5 Rows ---")
print(tesla_data.head())

# ---------------------------------------------------------
# Question 2: Use Webscraping to Extract Tesla Revenue Data
# ---------------------------------------------------------
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text
soup = BeautifulSoup(html_data, "html.parser")

tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
tables = soup.find_all("table")

for row in tables[1].find("tbody").find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    revenue = col[1].text
    tesla_revenue = pd.concat([tesla_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})], ignore_index=True)

tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(',|\$', "", regex=True)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

print("\n--- Question 2: Tesla Revenue Last 5 Rows ---")
print(tesla_revenue.tail())